# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kareemokeil/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [5]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/kareemokeil/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [7]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [8]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [9]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [10]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [11]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [12]:
# Experiment 1 — max_depth=3
tree_depth_3 = DecisionTreeClassifier(
    max_depth=3,
    class_weight="balanced",
    random_state=42
)

tree_depth_3.fit(X, y)

tree_depth_3_score = tree_depth_3.predict_proba(X)[:, 1]

for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_depth_3_score, y, k)
    print(f"Precision@{k}: hand rule {hr:.3f} vs tree {tr:.3f}")

print("\nDepth-3 Tree:")
print(export_text(tree_depth_3, feature_names=features))

Precision@20: hand rule 0.900 vs tree 0.700
Precision@50: hand rule 0.680 vs tree 0.720

Depth-3 Tree:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0



In [13]:
print("engagement_rate" in df.columns)

True


### Experiment: Increasing Tree Depth

I increased the decision tree depth from 2 to 3 to test whether an additional level of splits could improve the ranking performance.

The depth-2 tree achieved a Precision@20 of 0.550 and a Precision@50 of 0.600, while the depth-3 tree achieved 0.700 and 0.720 respectively.

The depth-3 tree therefore improved Precision@50 from 0.600 to 0.720 and also improved Precision@20. However, the hand-written rule still performed better at Precision@20 (0.900 vs. 0.700).

The tree remained readable, although it became more complex. Its first split was on `impressions_90d`, followed by `content_age_days`, `ctr`, and `avg_position`.

This suggests that allowing a slightly deeper tree helped the model capture additional interactions between the features that the simple hand-written rule could not capture.

In [15]:
# Experiment 2
features_alt = [
    "content_age_days",
    "days_since_last_update",
    "avg_position",
    "ctr",
    "word_count",
    "engagement_rate"
]

X_alt = (
    df[features_alt]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

tree_alt = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree_alt.fit(X_alt, y)

tree_alt_score = tree_alt.predict_proba(X_alt)[:, 1]

for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_alt_score, y, k)
    print(f"Precision@{k}: hand rule {hr:.3f} vs tree {tr:.3f}")

print("\nAlternative Feature Tree:")
print(export_text(tree_alt, feature_names=features_alt))

Precision@20: hand rule 0.900 vs tree 0.850
Precision@50: hand rule 0.680 vs tree 0.700

Alternative Feature Tree:
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0



### Experiment 2 — Alternative Features

I replaced `impressions_90d` with `engagement_rate` and trained another depth-2 decision tree using the alternative feature set.

**Results:**

- Precision@20: Hand rule **0.900** vs Tree **0.850**
- Precision@50: Hand rule **0.680** vs Tree **0.700**

The hand rule performs better at Precision@20, while the alternative-feature tree slightly outperforms it at Precision@50.

The learned tree selected `avg_position` as its first split, followed by `content_age_days`. This suggests that, with `impressions_90d` removed, the model found search position and content age to be the most useful signals for identifying declining pages.

The tree remained simple and readable, which makes its decision logic easy to interpret.

In [19]:
# Experiment 3 — Client-Holdout Validation

from sklearn.tree import DecisionTreeClassifier, export_text

# Features used by the original depth-2 tree
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

# Prepare features
X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining_label"].values

# 1. Split by client

clients = df["client_id"].dropna().unique()

rng = np.random.RandomState(42)
rng.shuffle(clients)

split_point = int(len(clients) * 0.8)

train_clients = clients[:split_point]
test_clients = clients[split_point:]

train_mask = df["client_id"].isin(train_clients)
test_mask = df["client_id"].isin(test_clients)

X_train = X.loc[train_mask]
y_train = y[train_mask]

X_test = X.loc[test_mask]
y_test = y[test_mask]

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Train pages:", len(X_train))
print("Test pages:", len(X_test))

# Verify that no client appears in both sets
print(
    "Client overlap:",
    len(set(train_clients) & set(test_clients))
)

# 2. Train the tree ONLY on training clients

tree_holdout = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

tree_holdout.fit(X_train, y_train)

# 3. Score ONLY the test clients

tree_test_score = tree_holdout.predict_proba(X_test)[:, 1]

hand_rule_test_score = df.loc[
    test_mask, "hand_rule_score"
].values

# 4. Compare Hand Rule vs Tree

for k in (20, 50):
    hr = precision_at_k(
        hand_rule_test_score,
        y_test,
        k
    )

    tr = precision_at_k(
        tree_test_score,
        y_test,
        k
    )

    print(
        f"Precision@{k}: "
        f"hand rule {hr:.3f} vs "
        f"tree {tr:.3f}"
    )

# 5. Read the tree

print("\nClient-Holdout Tree:")
print(
    export_text(
        tree_holdout,
        feature_names=features
    )
)

Train clients: 25
Test clients: 7
Train pages: 22389
Test pages: 7611
Client overlap: 0
Precision@20: hand rule 0.650 vs tree 0.650
Precision@50: hand rule 0.500 vs tree 0.600

Client-Holdout Tree:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 310.50
|   |   |--- class: 1
|   |--- content_age_days >  310.50
|   |   |--- class: 0



### Experiment — Client-Holdout Validation

I evaluated the hand-written rule and the depth-2 decision tree using a client-aware train/test split. The training set contained 25 clients and the test set contained 7 clients, with zero client overlap.

On the held-out clients, both methods achieved a Precision@20 of 0.650. At Precision@50, the decision tree achieved 0.600 compared with 0.500 for the hand rule.

This suggests that the tree generalizes slightly better at a deeper part of the ranking, while the hand rule performs equally well at the very top. The result is more meaningful than the earlier in-sample comparison because the model was evaluated on clients it had never seen during trainin

### Experiment Results

I ran three experiments to compare the hand-written rule with decision trees.

#### Experiment 1 — Increase `max_depth` to 3

The deeper tree improved Precision@50 compared with the original depth-2 tree.

| Model | Precision@20 | Precision@50 |
|---|---:|---:|
| Hand Rule | 0.900 | 0.680 |
| Depth-3 Tree | 0.700 | **0.720** |

The depth-3 tree achieved better Precision@50, but the hand rule was still better at Precision@20. The tree remained readable, although it introduced more decision splits.

#### Experiment 2 — Alternative Features

I removed `impressions_90d` and added `engagement_rate` to see whether the model would discover a different signal.

| Model | Precision@20 | Precision@50 |
|---|---:|---:|
| Hand Rule | **0.900** | 0.680 |
| Alternative Feature Tree | 0.850 | **0.700** |

The alternative tree selected `avg_position` as its first split, followed by `content_age_days`. This suggests that page ranking and content age provided useful predictive signals even without `impressions_90d`.

#### Experiment 3 — Client-Holdout Validation

Finally, I evaluated the model on clients that were completely excluded from training.

- Train clients: 25
- Test clients: 7
- Client overlap: 0
- Train pages: 22,389
- Test pages: 7,611

| Model | Precision@20 | Precision@50 |
|---|---:|---:|
| Hand Rule | 0.650 | 0.500 |
| Client-Holdout Tree | 0.650 | **0.600** |

The tree maintained an advantage at Precision@50 even on unseen clients. However, the gap was smaller than in the in-sample experiments, showing why client-holdout validation is important.

### Conclusion

The experiments showed that the hand-written rule is very strong at the very top of the ranking, especially at Precision@20. However, increasing the tree depth and changing the feature set allowed the model to discover additional patterns and improve Precision@50.

Most importantly, the client-holdout experiment showed that the improvement can still exist on clients the model has never seen, although it becomes smaller.

The `trend_pct` experiment also demonstrated why leakage must be avoided: adding a feature that directly determines the target produced a misleading Precision@50 of **1.000**. Therefore, the final model should only use signals that would genuinely be available before the outcome is known.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.